# Cytosolic Small-Molecule Buffering Analysis

This notebook investigates whether cytosolic small molecules (GSH, ATP, etc.) can account for
the **deficit** between total cellular metal measured by ICP-MS and the metal accounted for
by the proteome model in Figure 1.

## Rationale

The reviewer noted that the absence of *free* (hydrated) metal ions does not mean the absence
of an *exchangeable* labile pool: small molecules such as glutathione (GSH) and ATP buffer
transition metals at µM concentrations in the cytosol (Simm et al. 2021, JBIC).

The analysis proceeds in three steps (matching the todo list in the plan):

1. **Deficit calculation** – compute `deficit_uM = total_ICP − protein_sim`, converted to µM
   using simulation-derived cell volumes.
2. **Approach 1** – given the deficit and known small-molecule concentrations, estimate the
   minimum effective Kd a hypothetical single ligand would need to sequester the observed deficit.
3. **Approach 2** – use published stability constants to predict the labile pool and compare to
   the deficit.

In [49]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from wholecell.utils import units
from ecoli.library.schema import bulk_name_to_idx

os.makedirs("buffer_figures", exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.dpi": 120,
})

## 1. Load metals plot data

Columns:
- `Atoms/cell (experiment)` – ICP-MS total cellular metal  
- `Atoms/cell (simulation)` – model prediction (protein-bound metal only)  
- `Medium` – `"Rich"` or `"Minimal"`

In [2]:
df = pd.read_csv("data/metals_plot_data.csv")

# Strip stray quotes introduced by the quoting style in the CSV
df["Element"] = df["Element"].str.replace('"', "").str.strip()
df["Medium"]  = df["Medium"].str.replace('"', "").str.strip()

df

,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium
0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich
1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich
2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich
3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich
4,CO,443.558908,100.000000,0.000000,0.000000,Rich
5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich
6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich
7,V,249.432594,1000.000000,0.000000,0.000000,Rich
8,CR,3414.351358,1000.000000,0.000000,0.000000,Rich
9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal


## 2. Compute the deficit

$$
\text{deficit}_{\text{atoms}} = N_{\text{ICP}} - N_{\text{sim}}
$$

In [3]:
df["deficit_atoms"] = df["Atoms/cell (experiment)"] - df["Atoms/cell (simulation)"]

# Propagate uncertainties in quadrature
df["deficit_atoms_err"] = np.sqrt(
    df["Atoms/cell (experiment), stddev"]**2 +
    df["Atoms/cell (simulation), stddev"]**2
)

# ---- From final_paper_validation.ipynb: manually copied over the
# average cell volumes per condition used for the conversion to µM ----
cell_volume_fL = {
    "Rich":    3.073264,
    "Minimal": 1.44983
} # units in fL

COUNTS_UNITS = units.mol
VOLUME_UNITS = units.L
CONC_UNITS = COUNTS_UNITS / VOLUME_UNITS
micro_molar = units.umol/units.L

def atoms_to_uM(atoms_per_cell: float, volume_fL: float) -> float:
    """Convert atoms/cell to µM given cell volume in femtolitres."""
    n_avogadro = 6.02214076e23 / COUNTS_UNITS
    volume_L = VOLUME_UNITS * volume_fL * 1e-15
    molar    = atoms_per_cell / (n_avogadro * volume_L)

    return molar.asUnit(micro_molar)  # µM

In [4]:
# Convert to µM using per-condition cell volume
df["cell_volume_fL"] = df["Medium"].map(cell_volume_fL)
df["deficit_uM"]     = df.apply(
    lambda r: atoms_to_uM(r["deficit_atoms"], r["cell_volume_fL"]), axis=1
)
df["deficit_uM_err"] = df.apply(
    lambda r: atoms_to_uM(r["deficit_atoms_err"], r["cell_volume_fL"]), axis=1
)

# Also convert total and sim to µM for later use
df["total_uM"] = df.apply(
    lambda r: atoms_to_uM(r["Atoms/cell (experiment)"], r["cell_volume_fL"]), axis=1
)
df["sim_uM"] = df.apply(
    lambda r: atoms_to_uM(r["Atoms/cell (simulation)"], r["cell_volume_fL"]), axis=1
)

In [5]:
# Focus on the six biologically relevant transition metals
TARGET_METALS = ["FE", "ZN", "MN", "CU", "MO", "NI"]
df_metals = df[df["Element"].isin(TARGET_METALS)].copy()

display_cols = [
    "Element", "Medium",
    "total_uM", "sim_uM",
    "deficit_atoms", "deficit_uM", "deficit_uM_err",
]
df_metals[display_cols].sort_values(["Element", "Medium"]).reset_index(drop=True)

,Element,Medium,total_uM,sim_uM,deficit_atoms,deficit_uM,deficit_uM_err
0,CU,Minimal,8.991643832170565 [umol/L],9.728728786244183 [umol/L],-643.554795,-0.7370849540736187 [umol/L],1.5124513402886703 [umol/L]
1,CU,Rich,47.31922772286078 [umol/L],8.218817288590834 [umol/L],72365.586663,39.10041043426994 [umol/L],15.878535219571317 [umol/L]
2,FE,Minimal,206.1352859968846 [umol/L],306.48925646024685 [umol/L],-87619.857836,-100.35397046336227 [umol/L],34.29750126394372 [umol/L]
3,FE,Rich,328.0502349398715 [umol/L],267.2866391011376 [umol/L],112459.005207,60.763595838733856 [umol/L],71.5937961292907 [umol/L]
4,MN,Rich,25.25641070542673 [umol/L],37.49721065968548 [umol/L],-22654.817688,-12.240799954258755 [umol/L],7.270888323879342 [umol/L]
5,MO,Minimal,4.56223543910277 [umol/L],2.50833222258943 [umol/L],1793.279399,2.05390321651334 [umol/L],1.1876101771072853 [umol/L]
6,MO,Rich,5.833828296242571 [umol/L],1.861362602180166 [umol/L],7352.091890,3.9724656940624055 [umol/L],0.5807557002002923 [umol/L]
7,NI,Minimal,4.329449437680053 [umol/L],0.7835023416485402 [umol/L],3095.994897,3.545947096031513 [umol/L],1.1543519430757219 [umol/L]
8,NI,Rich,1.7507750117177838 [umol/L],0.694222832978171 [umol/L],1955.427511,1.0565521787396126 [umol/L],0.5486956448007573 [umol/L]
9,ZN,Minimal,228.84709374710573 [umol/L],246.2679884360221 [umol/L],-15210.323109,-17.420894688916388 [umol/L],65.32513329550545 [umol/L]


## 3. First Approach: Use exisitng small molecule concentration and $K_D$ reported in literature to estimate buffer concentration.

$$
[\text{Buffered}_{metal}] = \sum{K_A [\text{Metal}] [\text{Buffer}_i]}
$$

### load sim

In [6]:
import dill
def load_sim(
        folder_path:str,
):
    """ This function is meant to load an output of a simulation in timeseries form.
        Note: This is not designed for parquet output format.
    """
    # --- Load Sim ---
    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [9]:
os.chdir(os.path.expanduser('~/dev/vEcoli'))
[fba, bulk, metabolism, output] = load_sim(
    folder_path="out/objective_weight/basal/homeostatic_only_2000_2026-01-09/"
)

### Iron (Fe)

In [53]:
# Small Molecules suggested by Hider et al 2011 BioMetals
buffer_mol = ['CIT[c]', 'GLUTATHIONE[c]', 'CYS[c]']

buffer_idx = bulk_name_to_idx(buffer_mol, metabolism.bulk_ids)
buffer_count = (
    pd.DataFrame(bulk[buffer_idx])
    .mean(axis=0)
    .to_frame("Atoms/cell (simulation)")  # converts Series → DataFrame with named column
)
buffer_count.index = buffer_mol
buffer_count["sim_uM (umol/L)"] = buffer_count.apply(
    lambda r: atoms_to_uM(r["Atoms/cell (simulation)"], cell_volume_fL['Minimal']).asNumber(), axis=1
)
buffer_count

,Atoms/cell (simulation),sim_uM (umol/L)
CIT[c],1.520994e+06,1742.045671
GLUTATHIONE[c],9.747624e+06,11164.281889
CYS[c],1.798668e+04,20.600754


In [58]:
# Experimental available metal concentration for iron (FE+2) is 1.9(±0.2) × 10–6 M



,Atoms/cell (simulation),sim_uM (umol/L)
FE+2[c],2.759327e+06,3160.350458
FE+3[c],0.000000e+00,0.000000


In [59]:
df

,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium,deficit_atoms,deficit_atoms_err,cell_volume_fL,deficit_uM,deficit_uM_err,total_uM,sim_uM
0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich,72365.586663,29387.402939,3.073264,39.10041043426994 [umol/L],15.878535219571317 [umol/L],47.31922772286078 [umol/L],8.218817288590834 [umol/L]
1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich,7352.091890,1074.841069,3.073264,3.9724656940624055 [umol/L],0.5807557002002923 [umol/L],5.833828296242571 [umol/L],1.861362602180166 [umol/L]
2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich,1955.427511,1015.505510,3.073264,1.0565521787396126 [umol/L],0.5486956448007573 [umol/L],1.7507750117177838 [umol/L],0.694222832978171 [umol/L]
3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich,42390.778943,91003.718818,3.073264,22.90448998939954 [umol/L],49.170923928408456 [umol/L],215.93280723540272 [umol/L],193.02831724600316 [umol/L]
4,CO,443.558908,100.000000,0.000000,0.000000,Rich,443.558908,100.000000,3.073264,0.23966274792109304 [umol/L],0.05403177426911084 [umol/L],0.23966274792109304 [umol/L],0.0 [umol/L]
5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich,-22654.817688,13456.689924,3.073264,-12.240799954258755 [umol/L],7.270888323879342 [umol/L],25.25641070542673 [umol/L],37.49721065968548 [umol/L]
6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich,112459.005207,132503.137455,3.073264,60.763595838733856 [umol/L],71.5937961292907 [umol/L],328.0502349398715 [umol/L],267.2866391011376 [umol/L]
7,V,249.432594,1000.000000,0.000000,0.000000,Rich,249.432594,1000.000000,3.073264,0.1347728561436677 [umol/L],0.5403177426911084 [umol/L],0.1347728561436677 [umol/L],0.0 [umol/L]
8,CR,3414.351358,1000.000000,0.000000,0.000000,Rich,3414.351358,1000.000000,3.073264,1.8448346185088806 [umol/L],0.5403177426911084 [umol/L],1.8448346185088806 [umol/L],0.0 [umol/L]
9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal,-87619.857836,29945.423893,1.449830,-100.35397046336227 [umol/L],34.29750126394372 [umol/L],206.1352859968846 [umol/L],306.48925646024685 [umol/L]


#### Mn

In [42]:
# Small Molecules suggested by Hider et al 2011 BioMetals
buffer_mol = ['CIT[c]']

mol_idx = bulk_name_to_idx(buffer_mol, metabolism.bulk_ids)
mol_count = (
    pd.DataFrame(bulk[mol_idx])
    .mean(axis=0)
    .to_frame("Atoms/cell (simulation)")  # converts Series → DataFrame with named column
)
mol_count.index = buffer_mol
mol_count["sim_uM"] = mol_count.apply(
    lambda r: atoms_to_uM(r["Atoms/cell (simulation)"], cell_volume_fL['Minimal']), axis=1
)

In [45]:
ligand_conc = mol_count["sim_uM"].values  # ligand concentrations in µ
deficit = df_metals[df_metals["Element"] == "FE"]["deficit_uM"].values[0]  # deficit in µM

In [48]:
df_metals

,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium,deficit_atoms,deficit_atoms_err,cell_volume_fL,deficit_uM,deficit_uM_err,total_uM,sim_uM
0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich,72365.586663,29387.402939,3.073264,39.10041043426994 [umol/L],15.878535219571317 [umol/L],47.31922772286078 [umol/L],8.218817288590834 [umol/L]
1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich,7352.091890,1074.841069,3.073264,3.9724656940624055 [umol/L],0.5807557002002923 [umol/L],5.833828296242571 [umol/L],1.861362602180166 [umol/L]
2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich,1955.427511,1015.505510,3.073264,1.0565521787396126 [umol/L],0.5486956448007573 [umol/L],1.7507750117177838 [umol/L],0.694222832978171 [umol/L]
3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich,42390.778943,91003.718818,3.073264,22.90448998939954 [umol/L],49.170923928408456 [umol/L],215.93280723540272 [umol/L],193.02831724600316 [umol/L]
5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich,-22654.817688,13456.689924,3.073264,-12.240799954258755 [umol/L],7.270888323879342 [umol/L],25.25641070542673 [umol/L],37.49721065968548 [umol/L]
6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich,112459.005207,132503.137455,3.073264,60.763595838733856 [umol/L],71.5937961292907 [umol/L],328.0502349398715 [umol/L],267.2866391011376 [umol/L]
9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal,-87619.857836,29945.423893,1.449830,-100.35397046336227 [umol/L],34.29750126394372 [umol/L],206.1352859968846 [umol/L],306.48925646024685 [umol/L]
10,CU,7850.676467,1000.000000,8494.231262,862.443333,Minimal,-643.554795,1320.533416,1.449830,-0.7370849540736187 [umol/L],1.5124513402886703 [umol/L],8.991643832170565 [umol/L],9.728728786244183 [umol/L]
12,NI,3780.077086,1000.000000,684.082189,125.737464,Minimal,3095.994897,1007.873955,1.449830,3.545947096031513 [umol/L],1.1543519430757219 [umol/L],4.329449437680053 [umol/L],0.7835023416485402 [umol/L]
13,ZN,199808.236064,55212.922173,215018.559173,14304.788166,Minimal,-15210.323109,57035.898690,1.449830,-17.420894688916388 [umol/L],65.32513329550545 [umol/L],228.84709374710573 [umol/L],246.2679884360221 [umol/L]


In [43]:
mol_count

,Atoms/cell (simulation),sim_uM
CIT[c],1.520994e+06,1742.045671407249 [umol/L]
GLUTATHIONE[c],9.747624e+06,11164.281888911546 [umol/L]
CYS[c],1.798668e+04,20.60075381478532 [umol/L]


In [25]:
mol_count

CITRATE[c]        0.000000e+00
GLUTATHIONE[c]    9.747624e+06
L-CYSTINE[c]      0.000000e+00
dtype: float64